# Quickstart: Downloading a real ProteinGym experiment

This notebook shows how to download a real ProteinGym substitution experiment and parse it with `dms_parser`.

We will:

1. Download the ProteinGym metadata table
2. Download the full substitution benchmark table
3. Select one real DMS assay
4. Extract the assay rows
5. Recover the WT sequence from the metadata
6. Build a standardized dataset with `dms_parser`

In [19]:
from pathlib import Path

import pandas as pd

from dms_parser import build_proteingym_dataset, read_table, write_table
from dms_parser.io import download_file

## 1. Define local directories

In [20]:
DATA_DIR = Path("data/proteingym")
DATA_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = DATA_DIR / "DMS_substitutions.csv"
benchmark_path = DATA_DIR / "DMS_substitutions.parquet"

## 2. Download ProteinGym metadata and benchmark data

In [21]:
metadata_url = "https://raw.githubusercontent.com/OATML-Markslab/ProteinGym/main/reference_files/DMS_substitutions.csv"
benchmark_url = "https://proteingym.s3.us-east-2.amazonaws.com/DMS_substitutions.parquet"

download_file(metadata_url, metadata_path, overwrite=False)
download_file(benchmark_url, benchmark_path, overwrite=False)

print(metadata_path)
print(benchmark_path)

data/proteingym/DMS_substitutions.csv
data/proteingym/DMS_substitutions.parquet


## 3. Load the metadata and the benchmark table

In [22]:
df_meta = read_table(metadata_path)
df_all = read_table(benchmark_path)

print("Metadata shape:", df_meta.shape)
print("Benchmark shape:", df_all.shape)

Metadata shape: (217, 46)
Benchmark shape: (2465767, 6)


In [23]:
df_meta.head()

,DMS_index,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,...,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type
0,DMS_sub_0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1_ZIKV_Sourisseau_2019.csv,A0A140D2T1_ZIKV,Virus,Zika virus (ZIKV),MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,3423,False,9576,...,A0A140D2T1_ZIKV_Sourisseau_growth_2019.csv,effect,1,mutant,A0A140D2T1_ZIKV_theta_0.01.npy,A0A140D2T1_ZIKV.pdb,291-794,0.1,NaN,OrganismalFitness
1,DMS_sub_1,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2_9HIV1_Haddox_2018.csv,A0A192B1T2_9HIV1,Virus,Human immunodeficiency virus 1,MRVKGIQMNSQHLLRWGIMILGMIMICSVAGNLWVTVYYGVPVWKD...,852,False,12577,...,A0A192B1T2_9HIV1_Haddox_2018.csv,fitness,1,mutant,A0A192B1T2_9HIV1_theta_0.01.npy,A0A192B1T2_9HIV1.pdb,1-852,0.1,NaN,OrganismalFitness
2,DMS_sub_2,A0A1I9GEU1_NEIME_Kennouche_2019,A0A1I9GEU1_NEIME_Kennouche_2019.csv,A0A1I9GEU1_NEIME,Prokaryote,Neisseria meningitidis,FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSA...,161,False,922,...,A0A1I9GEU1_NEIME_Kennouche_2019.csv,piliation_log2_ratio,1,mutants,A0A1I9GEU1_NEIME_theta_0.2.npy,A0A1I9GEU1_NEIME.pdb,1-161,0.1,NaN,Activity
3,DMS_sub_3,A0A247D711_LISMN_Stadelmann_2021,A0A247D711_LISMN_Stadelmann_2021.csv,A0A247D711_LISMN,Prokaryote,Listeria monocytogenes,MNINDLIREIKNKDYTVKLSGTDSNSITQLIIRVNNDGNEYVISES...,87,False,1653,...,A0A247D711_LISMN_Stadelmann_2021.csv,mean_prediction,1,mutant,A0A247D711_LISMN_b03_theta_0.2.npy,A0A247D711_LISMN.pdb,1-87,1.0,NaN,Activity
4,DMS_sub_4,A0A2Z5U3Z0_9INFA_Doud_2016,A0A2Z5U3Z0_9INFA_Doud_2016.csv,A0A2Z5U3Z0_9INFA,Virus,Influenza A virus (A/WSN/1933(H1N1)),MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...,565,False,10715,...,A0A2Z5U3Z0_9INFA_Doud_2016.csv,transformed_pref,1,mutant,A0A2Z5U3Z0_9INFA_theta_0.01.npy,A0A2Z5U3Z0_9INFA.pdb,1-565,0.1,NaN,OrganismalFitness


In [24]:
df_all.head()

,DMS_score,DMS_score_bin,mutated_sequence,target_seq,mutant,DMS_id
0,-0.533935,1.0,MRKLSDELLIESYFKCTEMNLNRDFIELIENEIKRRSLGHIISV,MRKLSDELLIESYFKATEMNLNRDFIELIENEIKRRSLGHIISV,A16C,SDA_BACSU_Tsuboyama_2023_1PV0
1,-2.151397,0.0,MRKLSDELLIESYFKDTEMNLNRDFIELIENEIKRRSLGHIISV,MRKLSDELLIESYFKATEMNLNRDFIELIENEIKRRSLGHIISV,A16D,SDA_BACSU_Tsuboyama_2023_1PV0
2,-0.870078,1.0,MRKLSDELLIESYFKETEMNLNRDFIELIENEIKRRSLGHIISV,MRKLSDELLIESYFKATEMNLNRDFIELIENEIKRRSLGHIISV,A16E,SDA_BACSU_Tsuboyama_2023_1PV0
3,-0.328954,1.0,MRKLSDELLIESYFKFTEMNLNRDFIELIENEIKRRSLGHIISV,MRKLSDELLIESYFKATEMNLNRDFIELIENEIKRRSLGHIISV,A16F,SDA_BACSU_Tsuboyama_2023_1PV0
4,-0.961885,1.0,MRKLSDELLIESYFKGTEMNLNRDFIELIENEIKRRSLGHIISV,MRKLSDELLIESYFKATEMNLNRDFIELIENEIKRRSLGHIISV,A16G,SDA_BACSU_Tsuboyama_2023_1PV0


## 4. Inspect available columns

In [25]:
print("Metadata columns:")
print(df_meta.columns.tolist())

print("\nBenchmark columns:")
print(df_all.columns.tolist())

Metadata columns:
['DMS_index', 'DMS_id', 'DMS_filename', 'UniProt_ID', 'taxon', 'source_organism', 'target_seq', 'seq_len', 'includes_multiple_mutants', 'DMS_total_number_mutants', 'DMS_number_single_mutants', 'DMS_number_multiple_mutants', 'DMS_binarization_cutoff', 'DMS_binarization_method', 'first_author', 'title', 'year', 'jo', 'region_mutated', 'molecule_name', 'selection_assay', 'selection_type', 'MSA_filename', 'MSA_start', 'MSA_end', 'MSA_len', 'MSA_bitscore', 'MSA_theta', 'MSA_num_seqs', 'MSA_perc_cov', 'MSA_num_cov', 'MSA_N_eff', 'MSA_Neff_L', 'MSA_Neff_L_category', 'MSA_num_significant', 'MSA_num_significant_L', 'raw_DMS_filename', 'raw_DMS_phenotype_name', 'raw_DMS_directionality', 'raw_DMS_mutant_column', 'weight_file_name', 'pdb_file', 'pdb_range', 'ProteinGym_version', 'raw_mut_offset', 'coarse_selection_type']

Benchmark columns:
['DMS_score', 'DMS_score_bin', 'mutated_sequence', 'target_seq', 'mutant', 'DMS_id']


## 5. Select one real DMS assay

We select one assay from the metadata and then subset the full benchmark table using its `DMS_id`.

In [26]:
df_meta[["DMS_id", "UniProt_ID", "target_seq"]].head(10)

,DMS_id,UniProt_ID,target_seq
0,A0A140D2T1_ZIKV_Sourisseau_2019,A0A140D2T1_ZIKV,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
1,A0A192B1T2_9HIV1_Haddox_2018,A0A192B1T2_9HIV1,MRVKGIQMNSQHLLRWGIMILGMIMICSVAGNLWVTVYYGVPVWKD...
2,A0A1I9GEU1_NEIME_Kennouche_2019,A0A1I9GEU1_NEIME,FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSA...
3,A0A247D711_LISMN_Stadelmann_2021,A0A247D711_LISMN,MNINDLIREIKNKDYTVKLSGTDSNSITQLIIRVNNDGNEYVISES...
4,A0A2Z5U3Z0_9INFA_Doud_2016,A0A2Z5U3Z0_9INFA,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...
5,A0A2Z5U3Z0_9INFA_Wu_2014,A0A2Z5U3Z0_9INFA,MKAKLLVLLYAFVATDADTICIGYHANNSTDTVDTILEKNVAVTHS...
6,A4_HUMAN_Seuma_2022,A4_HUMAN,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...
7,A4D664_9INFA_Soh_2019,A4D664_9INFA,MERIKELRDLMSQSRTREILTKTTVDHMAIIKKYTSGRQEKNPALR...
8,A4GRB6_PSEAI_Chen_2020,A4GRB6_PSEAI,MFKLLSKLLVYLTASIMAIASPLAFSVDSSGEYPTVSEIPVGEVRL...
9,AACC1_PSEAI_Dandage_2018,AACC1_PSEAI,MLRSSNDVTQQGSRPKTKLGGSSMGIIRTCRLGPDQVKSMRAALDL...


In [27]:
selected_row = df_meta.dropna(subset=["DMS_id", "target_seq"]).iloc[0]

dms_id = selected_row["DMS_id"]
wt_sequence = selected_row["target_seq"]

print("Selected DMS_id:", dms_id)
print("WT length:", len(wt_sequence))
print("WT preview:", wt_sequence[:60])

Selected DMS_id: A0A140D2T1_ZIKV_Sourisseau_2019
WT length: 3423
WT preview: MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRMVLAILAFLRFTAIK


## 6. Extract the corresponding experiment from the benchmark table

In [29]:
if "DMS_id" not in df_all.columns:
    raise KeyError("The benchmark table does not contain a 'DMS_id' column.")

df_experiment = df_all[df_all["DMS_id"] == dms_id].copy()

print("Experiment shape:", df_experiment.shape)
df_experiment.head()

Experiment shape: (9576, 6)


,DMS_score,DMS_score_bin,mutated_sequence,target_seq,mutant,DMS_id
738565,0.030269,0.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291A,A0A140D2T1_ZIKV_Sourisseau_2019
738566,0.048604,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291Y,A0A140D2T1_ZIKV_Sourisseau_2019
738567,0.093642,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291W,A0A140D2T1_ZIKV_Sourisseau_2019
738568,0.626747,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291V,A0A140D2T1_ZIKV_Sourisseau_2019
738569,1.762066,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291T,A0A140D2T1_ZIKV_Sourisseau_2019


## 7. Detect the relevant ProteinGym columns

ProteinGym substitution data typically uses:
- `mutant`
- `DMS_score`

We check them explicitly before building the dataset.

In [30]:
candidate_variant_cols = ["mutant", "variant"]
candidate_score_cols = ["DMS_score", "score"]

variant_col = next((c for c in candidate_variant_cols if c in df_experiment.columns), None)
score_col = next((c for c in candidate_score_cols if c in df_experiment.columns), None)

print("Detected variant column:", variant_col)
print("Detected score column:", score_col)

if variant_col is None or score_col is None:
    raise ValueError("Could not automatically detect ProteinGym variant/score columns.")

Detected variant column: mutant
Detected score column: DMS_score


## 8. Save the selected experiment locally

The builder works from a file path, so we save the selected assay as a compact CSV.

In [31]:
experiment_path = DATA_DIR / f"{dms_id}.csv"
write_table(df_experiment, experiment_path, index=False)

experiment_path

PosixPath('data/proteingym/A0A140D2T1_ZIKV_Sourisseau_2019.csv')

## 9. Build the standardized dataset with `dms_parser`

In [32]:
df_built = build_proteingym_dataset(
    input_path=experiment_path,
    score_col=score_col,
    variant_col=variant_col,
    wt_sequence=wt_sequence,
    add_relative_score=True,
    relative_method="log_ratio",
    relative_output_col="score_log_ratio",
    add_binary_label=True,
    delta=0.15,
    higher_is_better=True,
    binary_output_col="score_binary_like",
)

df_built.head()

,DMS_score,DMS_score_bin,mutated_sequence,target_seq,mutant,DMS_id,dataset_id,source,protein_id,gene,...,is_wildtype,n_mutations,score_raw,parsed_variant,parsed_is_wildtype,parsed_n_mutations,parsed_mutations,parsed_position,parsed_wt_aa,parsed_mut_aa
0,0.030269,0.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291A,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.030269,I291A,False,1,"[{'token': 'I291A', 'wt_aa': 'I', 'position': ...",291,I,A
1,0.048604,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291Y,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.048604,I291Y,False,1,"[{'token': 'I291Y', 'wt_aa': 'I', 'position': ...",291,I,Y
2,0.093642,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291W,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.093642,I291W,False,1,"[{'token': 'I291W', 'wt_aa': 'I', 'position': ...",291,I,W
3,0.626747,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291V,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.626747,I291V,False,1,"[{'token': 'I291V', 'wt_aa': 'I', 'position': ...",291,I,V
4,1.762066,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291T,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,1.762066,I291T,False,1,"[{'token': 'I291T', 'wt_aa': 'I', 'position': ...",291,I,T


## 10. Inspect standardized output

In [33]:
cols = [
    "variant",
    "score_raw",
    "status",
    "is_wildtype",
    "n_mutations",
    "mutated_sequence",
    "score_log_ratio",
    "score_binary_like",
]

available_cols = [c for c in cols if c in df_built.columns]
df_built[available_cols].head(10)

,variant,score_raw,status,is_wildtype,n_mutations,mutated_sequence
0,I291A,0.030269,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
1,I291Y,0.048604,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
2,I291W,0.093642,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
3,I291V,0.626747,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
4,I291T,1.762066,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
5,I291S,0.017235,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
6,I291R,0.008824,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
7,I291Q,0.064509,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
8,I291P,0.006903,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...
9,I291C,0.050461,OK,False,1,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...


## 11. Keep only successfully parsed rows

In [34]:
df_ok = df_built[df_built["status"] == "OK"].copy()

print("All rows:", len(df_built))
print("OK rows:", len(df_ok))

df_ok.head()

All rows: 9576
OK rows: 9576


,DMS_score,DMS_score_bin,mutated_sequence,target_seq,mutant,DMS_id,dataset_id,source,protein_id,gene,...,is_wildtype,n_mutations,score_raw,parsed_variant,parsed_is_wildtype,parsed_n_mutations,parsed_mutations,parsed_position,parsed_wt_aa,parsed_mut_aa
0,0.030269,0.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291A,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.030269,I291A,False,1,"[{'token': 'I291A', 'wt_aa': 'I', 'position': ...",291,I,A
1,0.048604,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291Y,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.048604,I291Y,False,1,"[{'token': 'I291Y', 'wt_aa': 'I', 'position': ...",291,I,Y
2,0.093642,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291W,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.093642,I291W,False,1,"[{'token': 'I291W', 'wt_aa': 'I', 'position': ...",291,I,W
3,0.626747,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291V,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,0.626747,I291V,False,1,"[{'token': 'I291V', 'wt_aa': 'I', 'position': ...",291,I,V
4,1.762066,1.0,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,MKNPKKKSGGFRIVNMLKRGVARVNPLGGLKRLPAGLLLGHGPIRM...,I291T,A0A140D2T1_ZIKV_Sourisseau_2019,None,proteingym,None,None,...,False,1,1.762066,I291T,False,1,"[{'token': 'I291T', 'wt_aa': 'I', 'position': ...",291,I,T


## 12. Summary

We downloaded:
- real ProteinGym metadata
- real ProteinGym substitution benchmark data

Then we:
- selected one assay by `DMS_id`
- recovered the WT sequence from the metadata
- parsed the experiment with `dms_parser`
- computed WT-relative transformed scores
- generated pseudo-binary labels